# The second best model we'll do
* bangbangbang

## O. Setup 

In [0]:
NOM_EQUIPE = "telecacaton"   # ← remplacez par le nom de votre équipe

# Ne touchez pas au reste
TABLE_PREDICTIONS = f"workspace.default.predictions_equipe_{NOM_EQUIPE}"
print(f"Votre table de prédictions : {TABLE_PREDICTIONS}")

In [0]:
#%sql GRANT MODIFY ON TABLE workspace.default.predictions_equipe_telecacaton TO `cyprien.mas@telecom-paris.fr`;
#GRANT MODIFY ON TABLE workspace.default.predictions_equipe_telecacaton TO `hugo.hennion@telecom-paris.fr`;
#GRANT MODIFY ON TABLE workspace.default.predictions_equipe_telecacaton TO `clement.pesquet@telecom-paris.fr`;

## 1. Model

### Import et Setup

In [0]:
%pip install lightgbm
#dbutils.library.restartPython()

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.types import LongType
import lightgbm as lgb

### Chargement des données

In [0]:

# COMMAND ----------
train_df = spark.table("workspace.default.histo_ventes_train")
test_df  = spark.table("workspace.default.histo_ventes_test")

# Colonnes utiles uniquement → réduit la mémoire au minimum
cols_utiles = ["semaine", "code_agence", "code_article", "quantite"]

print("⏳ Chargement train...")
train_pd = train_df.select(cols_utiles).toPandas()
print(f"✅ Train chargé : {len(train_pd):,} lignes")

print("⏳ Chargement test...")
test_pd = test_df.select(["semaine", "code_agence", "code_article"]).toPandas()
print(f"✅ Test chargé  : {len(test_pd):,} lignes")

# Parse semaines
def parse_semaine(df):
    df = df.copy()
    df["annee"]   = df["semaine"].str.split("-").str[0].astype(int)
    df["num_sem"] = df["semaine"].str.split("-").str[1].astype(int)
    df["week_id"] = df["annee"] * 100 + df["num_sem"]
    return df

train_pd = parse_semaine(train_pd)
test_pd  = parse_semaine(test_pd)

train_pd["is_test"] = 0
test_pd["is_test"]  = 1
test_pd["quantite"] = np.nan

all_df = pd.concat([train_pd, test_pd], ignore_index=True)
all_df = all_df.sort_values(["code_agence", "code_article", "week_id"]).reset_index(drop=True)

print(f"\nTotal : {len(all_df):,} lignes | {all_df.memory_usage(deep=True).sum() / 1e6:.0f} MB en mémoire")

### 2. Feature Engineering

In [0]:
# COMMAND ----------
def add_features(df):
    df  = df.copy()
    grp = ["code_agence", "code_article"]
    pair_key = df["code_agence"].astype(str) + "_" + df["code_article"].astype(str)

    # ── Lags temporels ────────────────────────────────────────────────────
    for lag in [1, 2, 4, 8, 13, 26, 52, 104]:
        df[f"lag_{lag}"] = df.groupby(grp)["quantite"].shift(lag)

    # ── Rolling means & std (shift(1) → pas de fuite) ────────────────────
    shifted = df.groupby(grp)["quantite"].shift(1)
    for window in [4, 8, 13, 26, 52]:
        df[f"roll_mean_{window}"] = (
            shifted.groupby(pair_key)
                   .transform(lambda x: x.rolling(window, min_periods=1).mean())
        )
        df[f"roll_std_{window}"] = (
            shifted.groupby(pair_key)
                   .transform(lambda x: x.rolling(window, min_periods=1).std())
        )

    # ── Taux de zéros glissants ───────────────────────────────────────────
    is_zero     = (df["quantite"] == 0).astype(float)
    shifted_zero = is_zero.groupby(pair_key).shift(1)
    for window in [26, 52]:
        df[f"zero_rate_{window}"] = (
            shifted_zero.groupby(pair_key)
                        .transform(lambda x: x.rolling(window, min_periods=1).mean())
        )

    # ── Tendance récente : pente sur 8 semaines ──────────────────────────
    df["trend_8"] = (
        df.groupby(grp)["quantite"].shift(1)
        - df.groupby(grp)["quantite"].shift(9)
    ) / 8

    # ── Ratio N-1 vs moyenne historique ──────────────────────────────────
    df["ratio_n1_vs_mean"] = df["lag_52"] / (df["lag_52"].groupby(pair_key).transform("mean") + 1e-5)

    # ── Stats globales par paire (calculées sur train uniquement) ─────────
    train_only = df[df["is_test"] == 0]

    pair_stats = (
        train_only.groupby(grp)["quantite"]
        .agg(pair_mean="mean", pair_median="median", pair_max="max", pair_count="count")
        .reset_index()
    )
    df = df.merge(pair_stats, on=grp, how="left")

    # ── Stats par (paire × numéro de semaine) — signal saisonnier fort ───
    sem_stats = (
        train_only.groupby(grp + ["num_sem"])["quantite"]
        .agg(sem_mean="mean", sem_max="max", sem_median="median")
        .reset_index()
    )
    df = df.merge(sem_stats, on=grp + ["num_sem"], how="left")

    # ── Stats par agence et par article ──────────────────────────────────
    agence_stats = (
        train_only.groupby("code_agence")["quantite"]
        .agg(agence_mean="mean", agence_median="median")
        .reset_index()
    )
    df = df.merge(agence_stats, on="code_agence", how="left")

    article_stats = (
        train_only.groupby("code_article")["quantite"]
        .agg(article_mean="mean", article_median="median")
        .reset_index()
    )
    df = df.merge(article_stats, on="code_article", how="left")

    # ── Nombre de semaines actives (quantite > 0) ─────────────────────────
    active_weeks = (
        train_only.groupby(grp)
        .apply(lambda x: (x["quantite"] > 0).sum())
        .reset_index()
        .rename(columns={0: "n_active_weeks"})
    )
    df = df.merge(active_weeks, on=grp, how="left")

    # ── Encodage saisonnalité ─────────────────────────────────────────────
    df["sin_sem"] = np.sin(2 * np.pi * df["num_sem"] / 52)
    df["cos_sem"] = np.cos(2 * np.pi * df["num_sem"] / 52)

    return df

all_df = add_features(all_df)
print(f"Features OK — shape : {all_df.shape}")

### 3. Split Train / Validation / Test

In [0]:
# COMMAND ----------
FEATURES = [
    # Lags
    "lag_1", "lag_2", "lag_4", "lag_8", "lag_13", "lag_26", "lag_52", "lag_104",
    # Rolling stats
    "roll_mean_4", "roll_mean_8", "roll_mean_13", "roll_mean_26", "roll_mean_52",
    "roll_std_4",  "roll_std_8",  "roll_std_13",  "roll_std_26",  "roll_std_52",
    # Zero rates & trend
    "zero_rate_26", "zero_rate_52", "trend_8", "ratio_n1_vs_mean",
    # Pair stats
    "pair_mean", "pair_median", "pair_max", "pair_count",
    # Semaine × paire
    "sem_mean", "sem_max", "sem_median",
    # Agence / article
    "agence_mean", "agence_median",
    "article_mean", "article_median",
    # Activité
    "n_active_weeks",
    # Temps
    "annee", "num_sem", "sin_sem", "cos_sem",
]

TARGET = "quantite"

train_mask = (all_df["is_test"] == 0) & (all_df["semaine"] < "2025-01")
val_mask   = (all_df["is_test"] == 0) & (all_df["semaine"].between("2025-01", "2025-26"))
test_mask  =  all_df["is_test"] == 1

X_train = all_df.loc[train_mask, FEATURES]
y_train = all_df.loc[train_mask, TARGET]
X_val   = all_df.loc[val_mask,   FEATURES]
y_val   = all_df.loc[val_mask,   TARGET]
X_test  = all_df.loc[test_mask,  FEATURES]

print(f"X_train : {X_train.shape} | X_val : {X_val.shape} | X_test : {X_test.shape}")

### 4. Entraînement LightGBM

In [0]:
# COMMAND ----------
def wape_eval(y_pred, dataset):
    y_true = dataset.get_label()
    wape   = np.sum(np.abs(y_pred - y_true)) / (np.sum(y_true) + 1e-10)
    return "wape", wape, False  # lower is better

dtrain = lgb.Dataset(X_train, label=y_train)
dval   = lgb.Dataset(X_val,   label=y_val, reference=dtrain)

params = {
    "objective":               "tweedie",
    "tweedie_variance_power":  1.2,    # 1.2 = moins agressif sur les zéros que 1.5
    "metric":                  "None",
    "learning_rate":           0.02,   # lent mais plus précis
    "num_leaves":              255,
    "min_child_samples":       10,
    "feature_fraction":        0.7,
    "bagging_fraction":        0.7,
    "bagging_freq":            1,
    "reg_alpha":               0.05,
    "reg_lambda":              0.5,
    "n_jobs":                  -1,
    "seed":                    42,
    "verbose":                 -1,
}

model = lgb.train(
    params,
    dtrain,
    num_boost_round = 3000,
    valid_sets      = [dtrain, dval],
    valid_names     = ["train", "val"],
    feval           = wape_eval,
    callbacks       = [
        lgb.early_stopping(stopping_rounds=75, min_delta=1e-4),
        lgb.log_evaluation(period=100),
    ],
)

print(f"\nMeilleure itération : {model.best_iteration}")

### 5. Validation & Score

In [0]:
# COMMAND ----------
val_preds_raw = model.predict(X_val, num_iteration=model.best_iteration)
val_preds     = np.clip(np.round(val_preds_raw), 0, None).astype(int)

wape_lgb = np.sum(np.abs(val_preds - y_val.values)) / (np.sum(y_val.values) + 1e-10)
print(f"WAPE LightGBM brut         : {wape_lgb:.4f}")

# ── Post-processing : forcer les zéros évidents ───────────────────────────
pair_zero_rate = (
    train_pd.groupby(["code_agence", "code_article"])["quantite"]
    .apply(lambda x: (x == 0).mean())
    .reset_index()
    .rename(columns={"quantite": "zero_rate_global"})
)

val_rows = all_df.loc[val_mask, ["semaine", "code_agence", "code_article"]].copy()
val_rows["quantite"] = val_preds
val_rows = val_rows.merge(pair_zero_rate, on=["code_agence", "code_article"], how="left")
val_rows["quantite"] = np.where(val_rows["zero_rate_global"] > 0.98, 0, val_rows["quantite"])

wape_post = np.sum(np.abs(val_rows["quantite"].values - y_val.values)) / (np.sum(y_val.values) + 1e-10)
print(f"WAPE après filtre zéros    : {wape_post:.4f}")

# ── Blend LightGBM + N-1 ─────────────────────────────────────────────────
train_lookup = train_pd.set_index(["semaine", "code_agence", "code_article"])["quantite"]

def get_n1_qty(semaine, agence, article):
    annee, sem = semaine.split("-")
    key = (f"{int(annee)-1}-{sem}", agence, article)
    return train_lookup.get(key, np.nan)

val_rows["quantite_n1"] = [
    get_n1_qty(r.semaine, r.code_agence, r.code_article)
    for r in val_rows.itertuples()
]

ALPHA = 0.80  # 80% LightGBM + 20% N-1
val_rows["quantite_blend"] = np.where(
    val_rows["quantite_n1"].notna(),
    ALPHA * val_rows["quantite"] + (1 - ALPHA) * val_rows["quantite_n1"],
    val_rows["quantite"]
)
val_rows["quantite_blend"] = np.clip(np.round(val_rows["quantite_blend"]), 0, None).astype(int)

wape_blend = np.sum(np.abs(val_rows["quantite_blend"].values - y_val.values)) / (np.sum(y_val.values) + 1e-10)
print(f"WAPE après blend N-1       : {wape_blend:.4f}")

print()
print("Benchmarks :")
print("  Baseline N-1        → ~1.387")
print("  Blend N-1 + mean    → ~1.259")
print("  Votre ancien modèle → 1.1029")

# ── Feature importance ────────────────────────────────────────────────────
feat_imp = pd.DataFrame({
    "feature":    FEATURES,
    "importance": model.feature_importance(importance_type="gain"),
}).sort_values("importance", ascending=False)
display(feat_imp.head(20))

### 6. Génération & Sauvegarde des Prédictions

In [0]:
# COMMAND ----------
test_preds_raw = model.predict(X_test, num_iteration=model.best_iteration)
test_preds     = np.clip(np.round(test_preds_raw), 0, None).astype(int)

test_rows = all_df.loc[test_mask, ["semaine", "code_agence", "code_article"]].copy()
test_rows["quantite"] = test_preds

# Post-processing zéros
test_rows = test_rows.merge(pair_zero_rate, on=["code_agence", "code_article"], how="left")
test_rows["quantite"] = np.where(test_rows["zero_rate_global"] > 0.98, 0, test_rows["quantite"])

# Blend N-1
test_rows["quantite_n1"] = [
    get_n1_qty(r.semaine, r.code_agence, r.code_article)
    for r in test_rows.itertuples()
]
test_rows["quantite"] = np.where(
    test_rows["quantite_n1"].notna(),
    ALPHA * test_rows["quantite"] + (1 - ALPHA) * test_rows["quantite_n1"],
    test_rows["quantite"]
)
test_rows["quantite"] = np.clip(np.round(test_rows["quantite"]), 0, None).astype(int)

print(f"Prédictions générées : {len(test_rows):,}  (attendu : 272 344)")

# Sauvegarde
predictions_spark = (
    spark.createDataFrame(test_rows[["semaine", "code_agence", "code_article", "quantite"]])
    .withColumn("code_agence",  F.col("code_agence").cast(LongType()))
    .withColumn("code_article", F.col("code_article").cast(LongType()))
    .withColumn("quantite",     F.col("quantite").cast(LongType()))
)

predictions_spark.write.mode("overwrite").saveAsTable(TABLE_PREDICTIONS)
print(f"✅ Prédictions sauvegardées dans : {TABLE_PREDICTIONS}")